In [1]:
import os
import base64
import json
import random
from openai import OpenAI
import anthropic
import numpy as np
import re
from tqdm import tqdm
import pandas as pd
import time
from dotenv import load_dotenv


# Ablation Study Configuration

In [2]:
# Ablation Study Configuration
# Choose which agents to enable/disable for the ablation study
ENABLE_VISUAL_AGENT = False      
ENABLE_LANGUAGE_AGENT = True    
ENABLE_HALLUCINATION_AGENT = False 

# Output file suffix for the ablation configuration
ablation_config = []
if ENABLE_VISUAL_AGENT:
    ablation_config.append("visual")
if ENABLE_LANGUAGE_AGENT:
    ablation_config.append("language")
if ENABLE_HALLUCINATION_AGENT:
    ablation_config.append("hallucination")

# Create name suffix based on enabled agents
config_suffix = "_".join(ablation_config)
print(f"Running with configuration: {config_suffix}")

Running with configuration: language


# Load dataset

In [3]:
# Set base directory using relative path
base_dir = os.path.join(os.getcwd(), "dataset", "pororo")
qa_json_path = os.path.join(base_dir, "qa.json")
description_csv_path = os.path.join(base_dir, "descriptions.csv")
gif_paths = {}

def load_dataset(qa_json_path, description_csv_path):
    """Load and process Pororo dataset"""
    try:
        with open(qa_json_path, "r") as f:
            qa_data = json.load(f)["PororoQA"]
        descriptions = pd.read_csv(description_csv_path)
        return qa_data, descriptions
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], pd.DataFrame()

# TODO increase questions
def get_random_questions(qa_data, max_questions=40, base_pattern="Pororo_ENGLISH1", seed=42):
    random.seed(seed)
    
    # Filter all eligible questions
    filtered_questions = [q for q in qa_data if base_pattern in q["video_name"]]
    
    # Group by unique (video_name, supporting_num) pairs to avoid duplicates
    unique_pairs = {}
    for q in filtered_questions:
        key = (q["video_name"], q["supporting_num"])
        if key not in unique_pairs:
            unique_pairs[key] = []
        unique_pairs[key].append(q)
    
    # Sample from unique pairs
    unique_keys = list(unique_pairs.keys())
    selected_keys = random.sample(unique_keys, min(max_questions, len(unique_keys)))
    
    # Get one question from each selected pair
    sampled_questions = []
    episode_counts = {}
    
    for key in selected_keys:
        question = random.choice(unique_pairs[key])
        sampled_questions.append(question)
        episode = question["video_name"]
        episode_counts[episode] = episode_counts.get(episode, 0) + 1
    
    # Group by season for display
    season_episodes = {
        "Pororo_ENGLISH1_1": [],
        "Pororo_ENGLISH1_2": [],
        "Pororo_ENGLISH1_3": []
    }
    
    for ep in episode_counts.keys():
        season = "_".join(ep.split("_")[:3])
        if season in season_episodes:
            season_episodes[season].append(ep)
    
    # Print statistics
    print(f"\nSelected {len(sampled_questions)} questions from {len(episode_counts)} episodes:")
    for season in sorted(season_episodes.keys()):
        season_eps = {ep: episode_counts[ep] for ep in season_episodes[season]}
        if season_eps:
            print(f"\n{season}:")
            for ep in sorted(season_eps.keys()):
                print(f"  {ep}: {season_eps[ep]} questions")
    
    return sampled_questions

def get_seeded_question(questions, gif_num, base_seed=42):
    """Get deterministic random question for a GIF"""
    if not questions:
        return None
    local_random = random.Random(base_seed + gif_num)
    sorted_questions = sorted(questions, key=lambda x: x["qid"])
    return local_random.choice(sorted_questions)

def encode_gif(gif_path):
    """Encode GIF file as base64 string"""
    try:
        if not os.path.exists(gif_path):
            print(f"Error: GIF not found at {gif_path}")
            return None
        with open(gif_path, "rb") as gif_file:
            image_base64 = base64.b64encode(gif_file.read()).decode('utf-8')
            return image_base64
    except Exception as e:
        print(f"Error encoding GIF: {e}")
        return None

# Load data
qa_data, descriptions = load_dataset(qa_json_path, description_csv_path)

# Get random sample of questions TODO increase number of questions
sampled_questions = get_random_questions(qa_data, max_questions=40)

# Group questions by supporting_num
grouped_questions = {}
for entry in sampled_questions:
    video_name = entry["video_name"]
    supporting_num = entry["supporting_num"]
    key = (video_name, supporting_num)
    if key not in grouped_questions:
        grouped_questions[key] = []
    grouped_questions[key].append(entry)

# Get unique pairs to process
gif_pairs = sorted(list(grouped_questions.keys()))
correct_count = 0
total_count = len(gif_pairs)

# Used to store question information for each GIF pair
question_data = {}  
for video_name, gif_num in gif_pairs:
    current_questions = grouped_questions[(video_name, gif_num)]
    if current_questions:
        entry = get_seeded_question(current_questions, int(gif_num))
        
        question = entry["question"]
        correct_idx = entry["correct_idx"]
        answers = [entry[f"answer{i}"] for i in range(5)]
        correct_answer = answers[correct_idx]
        qid = entry["qid"]

        question_data[(video_name, gif_num)] = {
            'entry': entry,
            'question': question,
            'correct_answer': correct_answer,
            'qid': qid
        }

        # gif path
        episode_parts = video_name.split("_")
        episode_folder = os.path.join(base_dir, "Scenes_Dialogues",
                                "_".join(episode_parts[:-1]),
                                video_name)
        gif_paths[(video_name, gif_num)] = os.path.join(episode_folder, f"{gif_num}.gif")

# Initialize results list
results_ablation = []


Selected 40 questions from 22 episodes:

Pororo_ENGLISH1_1:
  Pororo_ENGLISH1_1_ep1: 1 questions
  Pororo_ENGLISH1_1_ep10: 2 questions
  Pororo_ENGLISH1_1_ep11: 1 questions
  Pororo_ENGLISH1_1_ep12: 4 questions
  Pororo_ENGLISH1_1_ep13: 2 questions
  Pororo_ENGLISH1_1_ep2: 4 questions
  Pororo_ENGLISH1_1_ep5: 1 questions
  Pororo_ENGLISH1_1_ep6: 4 questions
  Pororo_ENGLISH1_1_ep9: 1 questions

Pororo_ENGLISH1_2:
  Pororo_ENGLISH1_2_ep10: 1 questions
  Pororo_ENGLISH1_2_ep2: 2 questions
  Pororo_ENGLISH1_2_ep5: 2 questions
  Pororo_ENGLISH1_2_ep8: 3 questions

Pororo_ENGLISH1_3:
  Pororo_ENGLISH1_3_ep1: 1 questions
  Pororo_ENGLISH1_3_ep11: 1 questions
  Pororo_ENGLISH1_3_ep12: 2 questions
  Pororo_ENGLISH1_3_ep13: 1 questions
  Pororo_ENGLISH1_3_ep2: 1 questions
  Pororo_ENGLISH1_3_ep3: 1 questions
  Pororo_ENGLISH1_3_ep4: 1 questions
  Pororo_ENGLISH1_3_ep5: 2 questions
  Pororo_ENGLISH1_3_ep7: 2 questions


# Agents Configuration

In [4]:
load_dotenv()
# Configuration
# MODEL_NAME = "claude-3-5-haiku-20241022"
MODEL_NAME = "gpt-4o-mini"

is_openai_model = not MODEL_NAME.startswith("claude-")

if is_openai_model:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"Using OpenAI model: {MODEL_NAME}")
else:
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    print(f"Using Anthropic model: {MODEL_NAME}")

# Agent Implementations

# Visual agent: handles image-related tasks, and outputs image description
def visual_agent(image_base64, question, max_retries=3, retry_delay=2):
    # If visual agent is disabled, return a basic placeholder
    if not ENABLE_VISUAL_AGENT:
        return "This is a cartoon image from Pororo."

    prompt = f"""
    As a visual analysis expert, carefully analyze the image and provide a concise visual description. Avoid speculation or assumptions beyond the visible content.
    Focus on the following aspects, as relevant to answering the question: {question}.

    1. Characters: Identify characters with distinctive features.
    2. Actions and Interactions: Describe what character is doing, including body posture and interactions.
    3. Facial Expressions and Emotions: Note visible facial expressions (e.g., happy, surprised, angry).
    4. Scene: Identify whether the scene is indoors or outdoors, and specify the environment.
    5. Objects: Mention relevant items, positions, colors, and sizes.
    6. Layout: Describe where characters and objects are located (e.g., left of, behind).
    7. Attributes and Colors: List visible colors and give exact counts where possible.
    8. Counts: Number of characters or repeated items.
    9. Movement: Describe motion or visual cues if any.
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url",
                            "image_url": {"url": f"data:image/gif;base64,{image_base64}"}}
                        ]
                    }],
                    max_tokens=1000,
                    temperature=0.1
                )
                visual_desc = completion.choices[0].message.content.strip()
                return visual_desc
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/gif",
                                "data": image_base64
                            }}
                        ]
                    }],
                    max_tokens=1000,
                    temperature=0.1,
                )
                visual_desc = completion.content[0].text.strip()
                return visual_desc

        except Exception as e:
            print(f"Visual agent attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    print("Error: Visual agent failed to process the image")
    return None

# Language agent: handles text-related tasks, and outputs initial predicted answer
def language_agent(question, image_base64, visual_desc, description, subtitles, max_retries=3, retry_delay=2):
    # If language agent is disabled but this function is still called, return None
    if not ENABLE_LANGUAGE_AGENT:
        return None

    # Note: visual_desc is always available regardless of whether visual agent is enabled or not
    # - If visual agent is enabled: visual_desc contains the generated description
    # - If visual agent is disabled: visual_desc contains "This is a cartoon image from Pororo."
    prompt = f"""
    As a language analysis expert for cartoon animations, provide a concise and accurate answer to the question based on the available information using EXACTLY ONE SENTENCE:

    Input:
    Question: {question}
    Scene Description: {description}
    Visual Description: {visual_desc}
    Subtitles: {subtitles}

    Guidelines:
    1. No explanations allowed.
    2. Response must be in English only. DO NOT include text in any other language.
    3. Avoid phrases like "based on ...", "according to..." or "the description prided..."   
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,  
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {
                                    "type": "image_url",
                                    "image_url":
                                        {"url": f"data:image/gif;base64,{image_base64}"}
                                }
                            ],
                        }
                    ],
                    max_tokens=150,
                    temperature=0.1,
                )
                initial_predicted_answer = completion.choices[0].message.content.strip().lower()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                             "source": {"type": "base64", "media_type": "image/gif", "data": image_base64}},
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1,
                )
                initial_predicted_answer = completion.content[0].text.strip().lower()

            # Extract first sentence
            sentences = re.split(r'[.!?]', initial_predicted_answer)
            first_sentence = sentences[0].strip()

            # Skip empty sentences
            if not first_sentence and len(sentences) > 1:
                first_sentence = next((s.strip() for s in sentences if s.strip()), "")

            return first_sentence

        except Exception as e:
            print(f"Language agent attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    print("Error: Language agent failed to generate an answer")
    return None

# Hallucination detection agent: implement as a Critic agent that evaluates and potentially corrects answers
def hallucination_agent(question, image_base64, initial_predicted_answer, visual_desc, description, subtitles, max_retries=3, retry_delay=2):
    # If hallucination agent is disabled, return the initial prediction
    if not ENABLE_HALLUCINATION_AGENT:
        return initial_predicted_answer

    # Handle case when language_agent failed or was disabled
    if initial_predicted_answer is None:
        return "unknown"
        
    # Skip if initial answer is very short (likely to be correct in its simplicity)
    if len(initial_predicted_answer.split()) <= 3:
        return initial_predicted_answer.lower()
        
    # Implementing Critic Agent based on "Critic-V: VLM Critics Help Catch VLM Errors in Multimodal Reasoning"
    prompt = f"""
    As a critical expert in cartoon analysis, your task is to evaluate and potentially improve the answer to a cartoon-related question.
    
    Input:
    Question: {question}
    Scene Description: {description}
    Dialogue/Subtitles: {subtitles}
    Image Context: {visual_desc}
    Proposed Answer: {initial_predicted_answer}
    
    CRITIC EVALUATION PROCEDURE:
    1. CAREFULLY analyze the question to identify exactly what information is being requested
    2. IDENTIFY key elements in the scene description and dialogue that specifically answer the question
    3. EVALUATE how well the proposed answer addresses the exact question asked
    4. CHECK for any factual inconsistencies between the proposed answer and the supporting materials
    5. Consider if the answer is unnecessarily complex, ambiguous, or contains irrelevant information
    
    DECISION FRAMEWORK:
    - For questions about SPECIFIC EVENTS: Focus on exactly what happened, with minimal interpretation
    - For questions about CHARACTER SPEECH: Prioritize exact quotes from the dialogue when possible
    - For questions about OBJECTS/ENTITIES: Be precise about what was visibly present
    - For YES/NO questions: Ensure the core yes/no part is clearly stated first
    
    RESPOND with one of the following:
    KEEP: The answer directly addresses the question with accurate information
    REVISE: [concise corrected answer] - If the answer needs focused improvement
    
    REVISION PRINCIPLES:
    - Prioritize CONCISENESS - remove unnecessary explanations or details
    - Ensure FACTUAL ACCURACY based on the provided context
    - Match the STYLE AND TONE of the reference answers (simple, direct statements)
    - For YES/NO questions, start with "yes" or "no" followed by minimal supporting detail
    """
    
    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,  
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url", 
                             "image_url": {"url": f"data:image/gif;base64,{image_base64}"}}
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1,
                )
                response = completion.choices[0].message.content.strip()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image", 
                             "source": {
                                 "type": "base64",
                                 "media_type": "image/gif",
                                 "data": image_base64
                             }}
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1,
                )
                response = completion.content[0].text.strip()
            
            # Process the response to extract the decision and potential revision
            if response.upper().startswith("KEEP"):
                # Keep original answer
                return initial_predicted_answer.lower()
            elif response.upper().startswith("REVISE:"):
                # Extract revised answer
                revised_answer = response[7:].strip()  # Remove "REVISE: " prefix
                
                # Take just the first sentence of the revision to maintain consistency with language agent
                sentences = re.split(r'[.!?]', revised_answer)
                first_sentence = sentences[0].strip().lower() if sentences else ""
                
                # Only use the revised answer if it's not empty and substantial
                if first_sentence and len(first_sentence) >= 5:
                    return first_sentence
                    
            # Default to original if format is unclear or revision is too short
            return initial_predicted_answer.lower()

        except Exception as e:
            print(f"Critic agent attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue
    
    # If all retry attempts fail, return the initial prediction
    return initial_predicted_answer.lower()

Using OpenAI model: gpt-4o-mini


# Calculate accuracy

In [5]:
def compute_accuracy(question, correct_answer, predicted_answer, max_retries=2, retry_delay=2):
    if correct_answer.lower().strip() == predicted_answer.lower().strip():
        return 1.0

    prompt = f"""
    Evaluate the accuracy of the predicted answer according to strict criteria below.

    Input:
    Question: {question}
    Correct Answer: {correct_answer}
    Predicted Answer: {predicted_answer}

    Evaluation Rules:
    1. Return ONLY a numeric score from [1.0, 0.75, 0.5, 0.25, 0.0].
    2. Focus PRIMARILY on semantic equivalence.

    Scoring Criteria:
    - 1.0: Contains the correct core information, even if phrased differently or with additional details
    - 0.75: Mostly correct but missing minor information or containing slight inaccuracies
    - 0.5: Partially correct - contains some correct elements but misses important aspects
    - 0.25: Slightly correct - has a small element of the correct answer but is mostly wrong
    - 0.0: Completely incorrect, contradicts the correct answer, or avoids answering

    Scoring Examples:
    - Example of Score 1.0 (Perfect match or semantic equivalence):
    Question: "how did pororo feel after seeing that the flower has wilted"
    Correct: "he was very upset"
    Predicted: "pororo felt sad after seeing that the flower had wilted"
    Score: 1.0 (Synonyms with same core meaning)

    - Example of Score 1.0 (Additional details):
    Question: "what does crong do when pororo says 'come here'"
    Correct: "crong runs away from pororo"
    Predicted: "when pororo says 'come here,' crong tries to run away again"
    Score: 1.0 (Contains core information with additional details)

    - Example of Score 0.75 (Mostly correct but missing or slightly inaccurate information):
    Question: "what did loopy propose to the group after telling them about the flower"
    Correct: "loopy proposed that they should ask her anything"
    Predicted: "loopy proposed to the group that they ask the magic flower questions to predict the future"
    Score: 0.75 (Core action correct but adds slight inaccuracy about asking the flower directly)

    - Example of Score 0.5 (Partially correct):
    Question: "what does pororo almost forget to leave with poby"
    Correct: "the broken camera piece"
    Predicted: "pororo almost forgets to leave with poby's precious camera"
    Score: 0.5 (Mentions camera but misses the specific detail that it's broken)

    - Example of Score 0.25 (Slightly correct):
    Question: "what does eddy ask pororo"
    Correct: "he asks pororo what are you doing"
    Predicted: "eddy asks crong why pororo is acting so urgently"
    Score: 0.25 (Wrong recipient but related to pororo's actions)

    - Example of Score 0.0 (Completely incorrect):
    Question: "what was crong playing with as pororo entered the house"
    Correct: "crong was playing with a snowboard"
    Predicted: "crong was not shown playing with anything"
    Score: 0.0 (Directly contradicts the correct answer)
    """
    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=150,
                    temperature=0.1
                )
                response = completion.choices[0].message.content.strip()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": prompt
                    }],
                    max_tokens=150,
                    temperature=0.1,
                )
                response = completion.content[0].text.strip()

            numeric_match = re.search(r'(1\.0|0\.75|0\.5|0\.25|0\.0)', response)
            if numeric_match:
                score = float(numeric_match.group(1))
            else:
                score = 0.0

            return score

        except Exception as e:
            print(f"Accuracy calculation attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
    # Return 0 if it can't parse the score
    return 0.0

# Evaluate model performance

In [6]:
try:
    # Load dataset
    qa_data, descriptions = load_dataset(qa_json_path, description_csv_path)    
    # Initialize counters
    correct_count = 0
    total_count = len(gif_pairs)

    # Process each video and GIF pair
    for video_name, gif_num in tqdm(gif_pairs, total=total_count):
        # Get question information
        if (video_name, gif_num) not in question_data:
            print(f"No question data found for {video_name} GIF {gif_num}")
            continue
            
        # Use retrieved question information
        q_info = question_data[(video_name, gif_num)]
        question = q_info['question']
        correct_answer = q_info['correct_answer']
        qid = q_info['qid']
        
        # Get gif path
        gif_path = gif_paths[(video_name, gif_num)]
        gif_directory = os.path.dirname(gif_path)
        subtitles_path = os.path.join(gif_directory, "subtitles.txt")
        
        # Load subtitles
        with open(subtitles_path, "r") as f:
            subtitles = f.read()

        # Get description
        description_rows = descriptions.loc[
            (descriptions.iloc[:, 0] == video_name) &
            (descriptions.iloc[:, 1] == int(gif_num))
        ]
        if description_rows.empty:
            print(f"Description for {video_name} GIF {gif_num} not found")
            continue

        descriptions_list = description_rows.iloc[:, 2].tolist()
        description = " ".join(descriptions_list)
        
        # Encode GIF to base64
        image_base64 = encode_gif(gif_path)
        if not image_base64:
            print(f"Error: Failed to encode GIF {gif_num}")
            continue
        
        # Multi-agent prediction process
        visual_desc = visual_agent(image_base64, question=question)
        if visual_desc is None:
            print(f"Error: Visual agent failed to process GIF {gif_num}")
            continue

        # Print the visual description when language agent is disabled
        if not ENABLE_LANGUAGE_AGENT:
            print(f"Visual Description: {visual_desc}")

        # Get initial prediction from language agent
        initial_predicted_answer = language_agent(question, image_base64, visual_desc, description, subtitles)

        # Handle case when language agent is disabled
        if initial_predicted_answer is None:
            if not ENABLE_LANGUAGE_AGENT:
                # Set default answer
                initial_predicted_answer = "unknown"
                # Directly set the final prediction
                predicted_answer = "unknown" 
                
                # Skip hallucination detection when language agent is disabled
            else:
                print(f"Error for question {qid} - Failed to generate answer")
                continue
        else:
            # Only call hallucination agent when language agent is enabled and generated an answer
            if ENABLE_HALLUCINATION_AGENT:
                final_answer = hallucination_agent(
                    question=question,
                    image_base64=image_base64,
                    initial_predicted_answer=initial_predicted_answer,
                    visual_desc=visual_desc,
                    description=description,
                    subtitles=subtitles
                )
                predicted_answer = final_answer if final_answer else initial_predicted_answer
            else:
                predicted_answer = initial_predicted_answer
        
        # Calculate accuracy - ensure question parameter is passed
        is_correct = 0
        if predicted_answer is not None:
            is_correct = compute_accuracy(question, correct_answer, predicted_answer)
        correct_count += is_correct
        
        # Store current result
        result = {
            'gif_num': gif_num,
            'video_name': video_name,
            'qid': qid,
            'question': question,
            'correct_answer': correct_answer,
            'predicted_answer': predicted_answer,
            'accuracy': is_correct
        }
        results_ablation.append(result)
        
        # Print debugging info
        print(f"\nVideo name: {video_name}")
        print(f"GIF number: {gif_num}")
        print(f"QID: {qid}")
        print(f"Question: {question}")
        print(f"Correct Answer: {correct_answer}")
        print(f"Predicted Answer: {predicted_answer}")
        print(f"Accuracy: {float(is_correct):.4f}")

    # Calculate overall accuracy
    average_accuracy = correct_count / total_count if total_count > 0 else 0
    print(f"\nAverage Accuracy: {average_accuracy:.4f}")

except Exception as e:
    print(f"Unexpected error in evaluation: {e}")
    average_accuracy = 0

  2%|▎         | 1/40 [00:04<02:55,  4.49s/it]


Video name: Pororo_ENGLISH1_1_ep1
GIF number: 14
QID: 383
Question: whtat does eddy ask pororo
Correct Answer: he asks pororo what are you doing
Predicted Answer: eddy asks crong why pororo is acting urgently
Accuracy: 0.2500


  5%|▌         | 2/40 [00:07<02:17,  3.62s/it]


Video name: Pororo_ENGLISH1_1_ep10
GIF number: 12
QID: 1100
Question: were eddy's friends interested seeing his new toy?
Correct Answer: yes, they ran happily towards the new toy.
Predicted Answer: yes, eddy's friends were interested in seeing his new toy
Accuracy: 1.0000


  8%|▊         | 3/40 [00:10<02:02,  3.32s/it]

...


Video name: Pororo_ENGLISH1_3_ep5
GIF number: 25
QID: 2333
Question: does the friends find pororo behind the snow man?
Correct Answer: the friends find pororo behind the snow man.
Predicted Answer: no, the friends do not find pororo behind the snowman
Accuracy: 0.0000


 98%|█████████▊| 39/40 [02:19<00:05,  5.56s/it]


Video name: Pororo_ENGLISH1_3_ep7
GIF number: 25
QID: 2425
Question: what does crong do when pororo says "come here"
Correct Answer: crong runs away from pororo
Predicted Answer: when pororo says "come here," crong tries to run away again
Accuracy: 1.0000


100%|██████████| 40/40 [02:22<00:00,  3.55s/it]


Video name: Pororo_ENGLISH1_3_ep7
GIF number: 47
QID: 2446
Question: what does poby say when invited to play
Correct Answer: he says "of course"
Predicted Answer: poby responds with "of course" when invited to play
Accuracy: 1.0000

Average Accuracy: 0.6875


# Save results

In [7]:
# Clean up ablation results to remove any existing average rows
results_ablation = [r for r in results_ablation if r['gif_num'] != 'Average']

# Get unique videos and questions
unique_videos = len(set(r['video_name'] for r in results_ablation))
unique_questions = len(set(r['qid'] for r in results_ablation))

# Add row numbers to each result
for i, result in enumerate(results_ablation, 1):
    result['row_num'] = i

# Add average accuracy as the last row
average_result = {
    'row_num': len(results_ablation) + 1,
    'video_name': f'Total Videos: {unique_videos}',
    'gif_num': 'Average',
    'qid': '',
    'question': f'Total Questions: {unique_questions}',
    'correct_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy
}
results_ablation.append(average_result)

# Define column order
column_order = [
    'row_num',
    'video_name', 
    'gif_num',
    'qid',
    'question',
    'correct_answer',
    'predicted_answer',
    'accuracy'
]

# Save results with configuration in filename using the new directory structure
safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')
results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)

# Create ablation subdirectory if it doesn't exist
os.makedirs(os.path.join(results_dir, "ablation"), exist_ok=True)

# Save to ablation subdirectory with configuration in filename
output_path = os.path.join(results_dir, "ablation", f'pororo_ablation_{config_suffix}_{safe_model_name}.csv')

# Check if the file exists and explicitly remove it
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Convert to DataFrame and save with error handling
try:
    results_df = pd.DataFrame(results_ablation)
    results_df = results_df[column_order]
    
    # Save with explicit file opening to ensure it closes properly
    results_df.to_csv(output_path, index=False)
    
    # Verify the file was created
    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")

Existing file removed: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/ablation/pororo_ablation_language_gpt_4o_mini.csv
Results successfully saved to: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/ablation/pororo_ablation_language_gpt_4o_mini.csv
